In [32]:
from openai import OpenAI
from langchain_openai import ChatOpenAI
from langchain.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from pydantic import Field, BaseModel
from langchain.globals import set_debug
from dotenv import load_dotenv
import os

In [34]:
# load_dotenv()
# api_key=os.getenv("OPENAI_API_KEY")

# Função utilizada para permitir ao desenvolvedor acompanhar toda a cadeia de pensamento do LangChain e "debugar" processos
set_debug(False)

# Criação de uma classe para estruturar o formato de saída da resposta
class Destino(BaseModel):
    cidade:str = Field("A cidade recomendada para visitar")
    motivo:str = Field("Motivo pelo qual é interessante visitar essa cidade")

class Restaurantes(BaseModel):
    cidade:str = Field("A cidade recomendada para visitar")
    restaurantes:str = Field("Restaurantes recomendados na cidade")

parser_destino = JsonOutputParser(pydantic_object=Destino)
parser_restaurantes = JsonOutputParser(pydantic_object=Restaurantes)

numero_dias = 7
numero_criancas = 2
atividade = "música"

prompt = f"Crie um roteiro de viagem de {numero_dias} dias, para uma família com {numero_criancas} crianças que gosta de {atividade}."

# cliente = OpenAI(api_key=api_key)

In [ ]:
cliente = OpenAI(
    api_key="lm-studio",
    base_url="http://127.0.0.1:1234/v1"
)

resposta = cliente.chat.completions.create(
    model="google/gemma-3-1b",
    messages=[
        {
            "role":"system",
            "content": "Você é um assistente de roteiro de viagens."
        },
        {
            "role":"user",
            "content": prompt
        }
    ],
    temperature=0.0
)

In [ ]:
print(resposta.choices[0].message.content)

## Roteiro de Viagem de 7 Dias para Família: Música e Aventura!

**Público-alvo:** Família com 2 crianças (idade aproximada: 6-12 anos) apaixonadas por música.

**Estilo:** Combinação de cultura, natureza, experiências musicais e momentos de relaxamento.

**Orçamento:** Variável, dependendo das escolhas de hospedagem, alimentação e atividades. (Estimativa: R$ 7.000 - R$ 12.000 por família)

**Destino:** Uma combinação de cidades com forte tradição musical e paisagens deslumbrantes na América do Sul. Sugestão: **Buenos Aires, Argentina & Mendoza, Argentina**.

**Dia 1: Chegada em Buenos Aires – Imersão Musical**

*   Chegada no Aeroporto Internacional de Ezeiza (EZE), Buenos Aires.
*   Traslado para o hotel (recomenda-se um hotel bem localizado, próximo à Avenida Corrientes).
*   Tarde livre para explorar a região da Plaza de Mayo e o bairro histórico de La Boca.
*   Jantar em um restaurante tradicional argentino com música ao vivo (ex: tango!).

**Dia 2: Tango, Arte e Sabores Argentino

In [ ]:
# Criação de um Template de prompt para estruturação das respostas do modelo

prompt_template = PromptTemplate(
    template="""
       Crie um roteiro de viagem de {dias} dias,
       para uma família com {numero_criancas} crianças,
       que gostam de {atividade}. 
    """
)

prompt = prompt_template.format(
    dias = numero_dias,
    numero_criancas = numero_criancas,
    atividade = atividade
)

In [33]:
# Criação do modelo utilizando LangChain

modelo = ChatOpenAI(
    base_url="http://127.0.0.1:1234/v1",
    api_key="lm-studio",
    model="google/gemma-4-e4b",
    temperature=0.5
)

resposta = modelo.invoke(prompt)
print(resposta.content)

KeyboardInterrupt: 

In [13]:
# Prompt Template utilizando LCEL

prompt_template = PromptTemplate(
    template="""
    Sugira uma cidade dado o meu interesse por {interesse}.
    """,
    input_variables=["interesse"]
)

# Na cadeia a seguir, separada pelo operador "|" (pipe), temos o template do prompt, o modelo a ser utilizado e a forma de saída da resposta do modelo (que neste caso)
# é uma string
cadeia = prompt_template | modelo | StrOutputParser()

resposta = cadeia.invoke(
    {
        "interesse": "praias"
    }
)

print(resposta)

Com certeza! Para te sugerir a melhor cidade para você, preciso de mais informações sobre seus interesses em praias. Vamos pensar juntos:

**1. Qual tipo de praia você gosta?**

*   **Praia tranquila e familiar:** Com águas calmas, ideal para famílias com crianças ou para quem busca relaxamento.
*   **Praia badalada e animada:** Com ondas fortes, vida noturna agitada e muitas opções de bares e restaurantes.
*   **Praia selvagem e isolada:** Acesso difícil, com paisagens naturais exuberantes e a sensação de estar longe do mundo.
*   **Praia com infraestrutura completa:** Com hotéis, restaurantes, lojas, etc., perto da praia.

**2. Qual o seu orçamento?**

*   **Econômico:** (Aproximadamente R$ 500 - R$ 1.500 por semana)
*   **Médio:** (Aproximadamente R$ 1.500 - R$ 3.000 por semana)
*   **Alto:** (Aproximadamente R$ 3.000+ por semana)

**3. Que tipo de ambiente você prefere?**

*   **Cidade com vida cultural:** Com museus, teatros, restaurantes renomados e eventos culturais.
*   **Cidad

In [ ]:
# Prompt Template utilizando LCEL com saída no formato JSON

prompt_template = PromptTemplate(
    template="""
    Sugira uma cidade dado o meu interesse por {interesse}.

    Meu objetivo é um lugar para relaxar, de preferência distante dos grandes centros.
    {formato_de_saida}
    """,
    input_variables=["interesse"],
    partial_variables={"formato_de_saida":parser_destino.get_format_instructions()}
)

# Na cadeia a seguir, separada pelo operador "|" (pipe), temos o template do prompt, o modelo a ser utilizado e a forma de saída da resposta do modelo (que neste caso)
# é uma string
cadeia = prompt_template | modelo | parser_destino

resposta = cadeia.invoke(
    {
        "interesse": "praias"
    }
)

print(resposta)

{'properties': {'cidade': {'default': 'Porto Seguro, Portugal', 'title': 'Cidade', 'type': 'string'}, 'motivo': {'default': 'Beleza natural e clima agradável', 'title': 'Motivo', 'type': 'string'}}}


In [36]:
modelo = ChatOpenAI(
    base_url="http://127.0.0.1:1234/v1",
    api_key="lm-studio",
    model="google/gemma-4-e2b",
    temperature=0.5
)

# Encadeamento de Prompts

prompt_destino = PromptTemplate(
    template="""
    Sugira uma cidade dado o meu interesse por {interesse}.
    {formato_de_saida}
    """,
    input_variables=["interesse"],
    partial_variables={"formato_de_saida":parser_destino.get_format_instructions()}
)

# Por funcionar em formato encadeado, o LangChain permite referenciar variáveis vindas de cadeias anteriores. Neste caso, referencia-se o campo "cidade"
# da cadeira de destino
prompt_restaurantes = PromptTemplate(
    template="""
    Sugira restaurantes populares entre locais em {cidade}.
    {formato_de_saida}
    """,
    partial_variables={"formato_de_saida":parser_restaurantes.get_format_instructions()}
)

prompt_cultural = PromptTemplate(
    template="Sugira atividades e locais culturais em {cidade}"
)

# Na cadeia a seguir, separada pelo operador "|" (pipe), temos o template do prompt, o modelo a ser utilizado e a forma de saída da resposta do modelo (que neste caso)
# é uma string
cadeia_destino = prompt_destino | modelo | parser_destino
cadeia_restaurantes = prompt_restaurantes | modelo | parser_restaurantes
cadeia_cultural = prompt_cultural | modelo | JsonOutputParser()

cadeia = (cadeia_destino | cadeia_restaurantes | cadeia_cultural)

resposta = cadeia.invoke(
    {
        "interesse": "praias"
    }
)

print(resposta)

OutputParserException: Invalid json output: O Rio de Janeiro é um verdadeiro tesouro cultural e natural! Oferece uma combinação incrível de paisagens deslumbrantes, história vibrante, arte e muita energia.

Aqui está uma sugestão de atividades e locais culturais imperdíveis, divididos por categorias:

---

### 🏞️ Natureza e Vistas Panorâmicas (Os Clássicos)

**1. Pão de Açúcar (Morro da Urca)**
*   **O que fazer:** Subir pelas duas bondes para ter vistas espetaculares da Baía de Guanabara, do Cristo Redentor e das praias. É um clássico absoluto.
*   **Dica:** Vá ao pôr do sol para uma vista mágica.

**2. Cristo Redentor (Corcovado)**
*   **O que fazer:** Visitar o monumento mais famoso do mundo. A subida (de trem ou van) oferece uma perspectiva única da cidade e da Floresta da Tijuca.
*   **Dica:** Chegue cedo para evitar grandes filas.

**3. Praias Icônicas**
*   **Copacabana:** O coração da vida carioca, ideal para caminhar no calçadão, observar o movimento e sentir a energia da cida